# Лабораторна робота 2. Аналіз даних. Бібліотека Pandas

**Виконала:** Сєднєва Уляна, група 4-10  
**Варіант:** 19  
**Репозиторій GitHub:** _посилання буде додано_

In [ ]:
# Підключення бібліотек
import io
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

---
# Завдання 1. ВВП країн (Вікіпедія)

Зчитати датасет ВВП (млн дол. США) за країнами з Вікіпедії: https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)

In [ ]:
# Зчитування таблиці з Вікіпедії
# (Вікіпедія відхиляє запити без заголовка User-Agent, тому завантажуємо сторінку через requests)
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'
html = requests.get(url, headers={'User-Agent': 'Mozilla/5.0 (student lab work)'}, timeout=30).text
tables = pd.read_html(io.StringIO(html), match='IMF')
gdp_raw = max(tables, key=len)        # основна таблиця - найбільша серед знайдених

# Багаторівневі заголовки (джерело + Forecast/Estimate/Year) об'єднуємо в один рядок
if isinstance(gdp_raw.columns, pd.MultiIndex):
    gdp_raw.columns = [' '.join(dict.fromkeys(str(x) for x in col)).strip()
                       for col in gdp_raw.columns]
print('Знайдено таблиць:', len(tables))
print('Стовпці основної таблиці:', list(gdp_raw.columns))

In [ ]:
# 1. Вивести перших 5 рядків
gdp_raw.head()

In [ ]:
# 2. Визначити розмір датасета
print('Розмір датасету (рядки, стовпці):', gdp_raw.shape)

In [ ]:
# 3. Визначити оптимальну кількість стовпців
# Для аналізу потрібні назва країни та три показники ВВП (МВФ, Світовий банк, ООН).
# Стовпці з роком оцінки (Year) лише вказують рік, тому їх не залишаємо.
country_col = gdp_raw.columns[0]

def is_year_col(c):
    # стовпець з роками: у назві є Year або майже всі значення - чотиризначні роки
    if 'year' in c.lower():
        return True
    s = gdp_raw[c].astype(str).str.replace(r'\[.*?\]', '', regex=True).str.strip()
    return s.str.fullmatch(r'(19|20)\d\d').mean() > 0.8

value_cols = []
for source in ['IMF', 'World Bank', 'United Nations']:
    cands = [c for c in gdp_raw.columns[1:] if source.lower() in c.lower() and not is_year_col(c)]
    if cands:
        value_cols.append(cands[0])
if len(value_cols) < 3:   # запасний варіант: перші три стовпці зі значеннями
    value_cols = [c for c in gdp_raw.columns[1:] if not is_year_col(c)][:3]

year_cols = [c for c in gdp_raw.columns[1:] if is_year_col(c)]
print('Стовпці зі значеннями ВВП:', value_cols)
for c in year_cols:
    print(f'Найчастіший рік у стовпці "{c}":', gdp_raw[c].mode().iloc[0])

gdp = gdp_raw[[country_col] + value_cols].copy()
print('\nОптимальна кількість стовпців:', gdp.shape[1])

In [ ]:
# 4. Змінити назви стовпців на більш зручні
gdp.columns = ['Country', 'MVF_2024', 'WorldBank_2023', 'OON_2022']
# прибираємо примітки виду [n 1] з назв країн
gdp['Country'] = gdp['Country'].astype(str).str.replace(r'\[.*?\]', '', regex=True).str.strip()
# рядок "World" - сума по всіх країнах, а не країна; для аналізу країн його видаляємо
world_row = gdp[gdp['Country'] == 'World']
gdp = gdp[gdp['Country'] != 'World'].reset_index(drop=True)
print('Видалено агрегований рядок World:', len(world_row))
gdp.head()

In [ ]:
# 5. Визначити тип даних
gdp.info()

In [ ]:
# 6. Замінити "—" на NaN, перевірити пропуски та замінити їх середнім значенням
num_cols = ['MVF_2024', 'WorldBank_2023', 'OON_2022']
for c in num_cols:
    gdp[c] = (gdp[c].astype(str)
              .str.replace(r'\[.*?\]', '', regex=True)   # примітки
              .str.replace(',', '', regex=False)           # роздільники тисяч
              .str.strip()
              .replace({'—': np.nan, '–': np.nan, '-': np.nan, 'nan': np.nan}))
    gdp[c] = pd.to_numeric(gdp[c], errors='coerce')

print('Типи після перетворення:\n', gdp.dtypes, '\n')
print('Кількість пропущених значень:\n', gdp[num_cols].isna().sum())

imputed_mask = gdp[num_cols].isna().any(axis=1)   # запам'ятовуємо країни з пропусками
gdp[num_cols] = gdp[num_cols].fillna(gdp[num_cols].mean())
print('\nКраїн, у яких були пропуски (замінено середнім):', int(imputed_mask.sum()))

In [ ]:
# 7. Ще раз перевірити наявність пропущених значень
print(gdp.isna().sum())

In [ ]:
# 8. Перевірити наявність дублікатів та видалити їх
print('Повних дублікатів рядків:', gdp.duplicated().sum())
print('Дублікатів назв країн:', gdp['Country'].duplicated().sum())
gdp = gdp.drop_duplicates().reset_index(drop=True)
print('Розмір після видалення дублікатів:', gdp.shape)

In [ ]:
# 9. Описова статистика
gdp.describe()

In [ ]:
# 10. Відхилення між MVF_2024 та WorldBank_2023 для кожної країни
gdp['Diff_MVF_WB'] = gdp['MVF_2024'] - gdp['WorldBank_2023']
gdp['AbsDiff'] = gdp['Diff_MVF_WB'].abs()
gdp['Diff_%'] = gdp['Diff_MVF_WB'] / gdp['WorldBank_2023'] * 100

real = gdp[~imputed_mask.values]                 # лише країни без заповнених пропусків
top_abs = real.nlargest(5, 'AbsDiff')[['Country', 'MVF_2024', 'WorldBank_2023', 'Diff_MVF_WB', 'Diff_%']]
top_rel = real.reindex(real['Diff_%'].abs().nlargest(5).index)[['Country', 'MVF_2024', 'WorldBank_2023', 'Diff_MVF_WB', 'Diff_%']]
print('Найбільші відхилення в абсолютних значеннях (млн дол.):')
display(top_abs)
print('Найбільші відхилення у відсотках:')
display(top_rel)
print(f"Відповідь: в абсолютних значеннях показники найбільше відрізняються у країнах "
      f"{', '.join(top_abs['Country'].head(3))}; у відносних - у {', '.join(top_rel['Country'].head(3))}.")

In [ ]:
# 11. Кореляція між MVF_2024, WorldBank_2023 та OON_2022
corr = gdp[num_cols].corr()
display(corr)
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().sort_values(ascending=False)
print(pairs)
print(f'Відповідь: найвища кореляція між {pairs.index[0][0]} та {pairs.index[0][1]} (r = {pairs.iloc[0]:.4f}).')

In [ ]:
# 12. Середнє значення для кожного стовпця та його зміна між роками
means = gdp[num_cols].mean()
print(means)
print(f"\nЗміна OON_2022 -> WorldBank_2023: {means['WorldBank_2023'] - means['OON_2022']:+,.0f} млн дол. "
      f"({(means['WorldBank_2023'] / means['OON_2022'] - 1) * 100:+.2f} %)")
print(f"Зміна WorldBank_2023 -> MVF_2024: {means['MVF_2024'] - means['WorldBank_2023']:+,.0f} млн дол. "
      f"({(means['MVF_2024'] / means['WorldBank_2023'] - 1) * 100:+.2f} %)")

In [ ]:
# 13. Стандартне відхилення показників для кожної країни
gdp['Std'] = gdp[num_cols].std(axis=1)
gdp['CV_%'] = gdp['Std'] / gdp[num_cols].mean(axis=1) * 100   # відносна варіативність
real = gdp[~imputed_mask.values]
print('Найбільше стандартне відхилення (абсолютна варіативність):')
display(real.nlargest(5, 'Std')[['Country'] + num_cols + ['Std']])
print('Найбільший коефіцієнт варіації (відносна варіативність):')
display(real.nlargest(5, 'CV_%')[['Country'] + num_cols + ['CV_%']])
print(f"Відповідь: найвища абсолютна варіативність - {real.loc[real['Std'].idxmax(), 'Country']}, "
      f"найвища відносна - {real.loc[real['CV_%'].idxmax(), 'Country']}.")

In [ ]:
# 14. Країни з найвищим та найнижчим показниками в кожному році
for c in num_cols:
    hi, lo = gdp.loc[gdp[c].idxmax()], gdp.loc[gdp[c].idxmin()]
    print(f"{c}: найвищий - {hi['Country']} ({hi[c]:,.0f}), найнижчий - {lo['Country']} ({lo[c]:,.0f})")

In [ ]:
# 15. Гістограма розподілу MVF_2024
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(gdp['MVF_2024'], bins=40, color='steelblue', edgecolor='white')
axes[0].set(title='Розподіл ВВП (MVF_2024)', xlabel='ВВП, млн дол. США', ylabel='Кількість країн')
axes[1].hist(np.log10(gdp['MVF_2024']), bins=30, color='darkorange', edgecolor='white')
axes[1].set(title='Той самий розподіл у логарифмічній шкалі', xlabel='log10(ВВП)', ylabel='Кількість країн')
plt.tight_layout()
plt.show()

q1, q3 = gdp['MVF_2024'].quantile([0.25, 0.75])
outl = gdp[gdp['MVF_2024'] > q3 + 1.5 * (q3 - q1)].sort_values('MVF_2024', ascending=False)
print(f"Медіана: {gdp['MVF_2024'].median():,.0f}, середнє: {gdp['MVF_2024'].mean():,.0f}")
print(f"Країн, що виділяються (вище Q3 + 1.5·IQR): {len(outl)}; найбільші: {', '.join(outl['Country'].head(5))}")

In [ ]:
# 16. Частка кожної країни в загальному значенні для кожного року
shares = gdp[['Country']].copy()
for c in num_cols:
    shares[c + '_share_%'] = gdp[c] / gdp[c].sum() * 100
shares['Зміна частки_2022→2024'] = shares['MVF_2024_share_%'] - shares['OON_2022_share_%']
top10 = shares.sort_values('MVF_2024_share_%', ascending=False).head(10)
display(top10)
print(f"Частка 10 найбільших економік у 2024: {top10['MVF_2024_share_%'].sum():.1f} %")
gain = shares.nlargest(3, 'Зміна частки_2022→2024')['Country'].tolist()
loss = shares.nsmallest(3, 'Зміна частки_2022→2024')['Country'].tolist()
print('Найбільше зросла частка:', ', '.join(gain))
print('Найбільше зменшилась частка:', ', '.join(loss))

In [ ]:
# 17. Зміни показників для країн за три роки
years = ['OON_2022', 'WorldBank_2023', 'MVF_2024']          # у хронологічному порядку
top = gdp.nlargest(10, 'MVF_2024').set_index('Country')[years]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for country, row in top.iterrows():
    axes[0].plot(['2022', '2023', '2024'], row.values / 1e6, marker='o', label=country)
axes[0].set(title='ВВП 10 найбільших економік', ylabel='трлн дол. США')
axes[0].legend(fontsize=8)
for country, row in top.iloc[2:].iterrows():
    axes[1].plot(['2022', '2023', '2024'], row.values / 1e6, marker='o', label=country)
axes[1].set(title='Без США та Китаю (для наочності)', ylabel='трлн дол. США')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

real = gdp[~imputed_mask.values]
growth = real[(real['WorldBank_2023'] > real['OON_2022']) & (real['MVF_2024'] > real['WorldBank_2023'])]
decline = real[(real['WorldBank_2023'] < real['OON_2022']) & (real['MVF_2024'] < real['WorldBank_2023'])]
print(f'Стабільне зростання (2022 < 2023 < 2024): {len(growth)} країн з {len(real)}')
print('  серед 10 найбільших:', ', '.join(c for c in top.index if c in set(growth['Country'])))
print(f'Стабільний спад (2022 > 2023 > 2024): {len(decline)} країн')
print('  приклади:', ', '.join(decline.nsmallest(5, 'MVF_2024')['Country'] if len(decline) else []))
print('  найбільші за розміром економіки зі спадом:', ', '.join(decline.nlargest(5, 'MVF_2024')['Country']))

### Висновок до завдання 1

Таблицю ВВП країн було зчитано з Вікіпедії та підготовлено до аналізу. З усіх стовпців залишено назву країни та три показники: МВФ, Світовий банк та ООН. Стовпці названо `MVF_2024`, `WorldBank_2023` і `OON_2022`, як вимагає методичка, хоча у Вікіпедії зараз розміщено новіші дані: прогноз МВФ на 2026 рік, оцінку Світового банку за 2025 рік та оцінку ООН за 2024 рік. Тому далі «2022 → 2024» означає порядок від найстарішої оцінки до найновішої. Агрегований рядок World видалено, оскільки він є сумою по всіх країнах.

Після перетворення в числовий формат виявлено пропуски: 30 у даних МВФ, 35 у даних Світового банку та 9 у даних ООН, загалом у 38 країнах. Їх замінено середнім значенням стовпця. Дублікатів не знайдено, тож для аналізу залишилося 221 країна.

Розподіл ВВП дуже нерівномірний. Медіана за даними МВФ становить близько 102 млрд дол., а середнє — близько 669 млрд дол., тобто в 6,5 раза більше. Отже, середнє значення сильно завищують кілька найбільших економік. На гістограмі майже всі країни потрапляють у перший інтервал, і лише в логарифмічній шкалі видно форму розподілу. За правилом Q3 + 1,5·IQR виділяються 16 країн, найбільші з них — США, Китай, Німеччина, Японія та Велика Британія. Найбільший ВВП за всіма трьома джерелами має США (32,6 трлн дол. за МВФ), найменший — Тувалу (65 млн дол.). Частка 10 найбільших економік становить 56,9 % світового ВВП, а частка однієї лише США — 22,1 %.

Показники трьох організацій майже однакові: коефіцієнти кореляції між ними близькі до 1, найвищий між МВФ і Світовим банком (r = 0,9985). В абсолютних значеннях оцінки найбільше розходяться для Китаю (2,05 трлн дол.), США та Німеччини, тобто для найбільших економік. У відсотках розходження найбільші для Бурунді (141,8 %), Туркменістану (66,7 %) та Замбії (42,8 %), де економічна статистика менш надійна. Аналогічно за стандартним відхиленням найбільш варіативний показник у США, а за коефіцієнтом варіації — у Бурунді (47,2 %).

Від найстарішої оцінки до найновішої середній ВВП зріс з 522 до 626 млрд дол. (+19,8 %), а потім до 669 млрд дол. (+6,9 %). Стабільне зростання за всіма трьома оцінками показали 157 країн зі 183, у яких не було пропусків, зокрема всі 10 найбільших економік. Стабільний спад спостерігався лише в 4 країнах: Ірані, Іраку, Катарі та Кувейті. Водночас частки США, Китаю та Індії у світовому ВВП трохи зменшилися: інші країни зростали швидше.

---
# Завдання 2. Титанік

In [ ]:
# Завантаження датасету Титанік з файлу titanic_5.csv
# (у Colab файл спочатку завантажують у панель "Файли" зліва або через вікно вибору нижче)
import os
fname = 'titanic_5.csv'
if not os.path.exists(fname):
    from google.colab import files
    files.upload()          # вибрати titanic_5.csv на комп'ютері
titanic = pd.read_csv(fname)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
print('Зчитано файл:', fname)

In [ ]:
# 1. Вивести перших 5 рядків
titanic.head()

In [ ]:
# 2. Визначити розмір датасета
print('Розмір датасету (рядки, стовпці):', titanic.shape)

In [ ]:
# 3. Визначити тип даних
titanic.info()

In [ ]:
# 4. Наявність пропущених значень і заміна
print('Пропущені значення до заміни:\n', titanic.isna().sum()[titanic.isna().sum() > 0], '\n')

# Числові пропуски (Age) замінюємо середнім значенням.
# Для текстових стовпців середнє не існує: Embarked заповнюємо найчастішим значенням (модою),
# а Cabin (77 % пропусків) не заповнюємо - він буде видалений у пункті 8.
print(f"Середній вік, яким заповнюємо пропуски: {titanic['Age'].mean():.2f}")
titanic['Age'] = titanic['Age'].fillna(titanic['Age'].mean())
titanic['Embarked'] = titanic['Embarked'].fillna(titanic['Embarked'].mode()[0])

In [ ]:
# 5. Ще раз перевірити наявність пропущених значень
print(titanic.isna().sum())

In [ ]:
# 6. Перевірити наявність дублікатів та видалити їх
print('Кількість дублікатів:', titanic.duplicated().sum())
titanic = titanic.drop_duplicates()
print('Розмір після перевірки:', titanic.shape)

In [ ]:
# 7. Описова статистика
titanic.describe()

In [ ]:
# 8. Видалити стовпчик Cabin
titanic = titanic.drop(columns='Cabin')
print('Стовпці:', list(titanic.columns))

In [ ]:
# 9. Сформувати датасет з обраними стовпцями
df = titanic[['Survived', 'Pclass', 'Sex', 'Age', 'Fare']].copy()
df.head()

In [ ]:
# 10. Замінити бінарні ознаки (Sex) на 0 і 1
print('Унікальні значення Sex:', df['Sex'].unique())
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
print('Після заміни:', df['Sex'].unique(), '(0 - чоловік, 1 - жінка)')

In [ ]:
# 11. Ще раз перевірити кількість пропущених даних
print(df.isna().sum())

In [ ]:
# 12. Вивести 5 перших рядків
df.head()

In [ ]:
# 13. Вивести 5 останніх рядків
df.tail()

In [ ]:
# 14. Відсоток виживання для кожної статі
surv_sex = df.groupby('Sex')['Survived'].mean().mul(100).rename({0: 'Чоловіки', 1: 'Жінки'})
print(surv_sex.round(2))
print(f"Відповідь: так, жінки виживали в {surv_sex['Жінки'] / surv_sex['Чоловіки']:.1f} раза частіше за чоловіків.")

In [ ]:
# 15. Відсоток виживання для кожного класу
surv_class = df.groupby('Pclass')['Survived'].mean().mul(100)
print(surv_class.round(2))
print(f'Відповідь: найвищий рівень виживання мав {surv_class.idxmax()}-й клас ({surv_class.max():.1f} %).')

In [ ]:
# 16. Середній вік тих, хто вижив, і тих, хто не вижив
age_surv = df.groupby('Survived')['Age'].mean().rename({0: 'Не вижили', 1: 'Вижили'})
print(age_surv.round(2))
print('Кореляція віку з виживанням:', round(df['Age'].corr(df['Survived']), 3))

In [ ]:
# 17. Рівень виживання за групами тарифів (квартилі Fare)
df['FareGroup'] = pd.qcut(df['Fare'], 4, labels=['Низький', 'Нижче середнього', 'Вище середнього', 'Високий'])
fare_surv = df.groupby('FareGroup', observed=True).agg(
    Межі_тарифу=('Fare', lambda s: f'{s.min():.1f}–{s.max():.1f}'),
    Пасажирів=('Survived', 'size'),
    Виживання_pct=('Survived', lambda s: s.mean() * 100))
fare_surv

In [ ]:
# 18. Середній тариф для кожного класу
fare_class = df.groupby('Pclass')['Fare'].mean()
print(fare_class.round(2))
print(f'Тариф 1-го класу більший за тариф 3-го у {fare_class[1] / fare_class[3]:.1f} раза.')

In [ ]:
# 19. Середній вік пасажирів для кожного класу
age_class = df.groupby('Pclass')['Age'].mean()
print(age_class.round(2))

In [ ]:
# 20. Гістограма розподілу віку для тих, хто вижив і не вижив
plt.figure(figsize=(9, 4.5))
bins = np.arange(0, 85, 5)
plt.hist(df[df['Survived'] == 0]['Age'], bins=bins, alpha=0.6, label='Не вижили', color='firebrick')
plt.hist(df[df['Survived'] == 1]['Age'], bins=bins, alpha=0.6, label='Вижили', color='seagreen')
plt.xlabel('Вік'); plt.ylabel('Кількість пасажирів')
plt.title('Розподіл віку пасажирів Титаніка')
plt.legend()
plt.show()

age_groups = pd.cut(df['Age'], [0, 12, 18, 30, 50, 80], labels=['0–12', '13–18', '19–30', '31–50', '51–80'])
print('Виживання за віковими групами, %:')
print(df.groupby(age_groups, observed=True)['Survived'].mean().mul(100).round(1))

In [ ]:
# 21. Відсоток виживання для кожної комбінації статі та класу
combo = (df.assign(Стать=df['Sex'].map({0: 'Чоловіки', 1: 'Жінки'}))
           .pivot_table(index='Стать', columns='Pclass', values='Survived', aggfunc='mean') * 100)
display(combo.round(1))
best = combo.stack().idxmax(); worst = combo.stack().idxmin()
print(f'Найвищий рівень виживання: {best[0].lower()} у {best[1]}-му класі ({combo.stack().max():.1f} %)')
print(f'Найнижчий: {worst[0].lower()} у {worst[1]}-му класі ({combo.stack().min():.1f} %)')

In [ ]:
# 22. Кореляція між числовими змінними
corr_t = df[['Survived', 'Pclass', 'Sex', 'Age', 'Fare']].corr()
display(corr_t.round(3))
print('Кореляція з Survived (за спаданням модуля):')
print(corr_t['Survived'].drop('Survived').sort_values(key=abs, ascending=False).round(3))

### Висновок до завдання 2

Датасет Titanic містить 891 пасажира та 12 ознак. Пропуски виявлено у трьох стовпцях: вік (177), каюта (687, тобто 77 %) та порт посадки (2). Пропущений вік замінено середнім значенням 29,70 року, порт посадки — найчастішим значенням. Стовпець Cabin видалено, оскільки більшість його значень відсутні. Дублікатів у даних немає. Для аналізу сформовано датасет зі стовпців Survived, Pclass, Sex, Age та Fare, а стать закодовано числами: 0 — чоловік, 1 — жінка.

Загалом вижило лише 38 % пасажирів. Найсильніше на виживання вплинула стать: жінки вижили у 74,2 % випадків, чоловіки — лише у 18,9 %, тобто в 3,9 раза рідше. Це підтверджує правило «спочатку жінки та діти». Другим за важливістю чинником був клас каюти: у 1-му класі вижили 63,0 % пасажирів, у 2-му — 47,3 %, у 3-му — лише 24,2 %. Зведена таблиця показує, що ці два чинники посилюють один одного: найвищий рівень виживання мали жінки 1-го класу (96,8 %), найнижчий — чоловіки 3-го класу (13,5 %).

Вартість квитка пов’язана з класом: середній тариф 1-го класу (84,15 дол.) у 6,2 раза більший, ніж 3-го (13,68 дол.). Тому з ростом тарифу зростає і виживання: від 19,7 % у групі з найдешевшими квитками до 58,1 % у групі з найдорожчими. Пасажири 1-го класу були в середньому старшими (37,1 року), ніж пасажири 3-го класу (26,4 року).

Вік вплинув на виживання слабко: середній вік тих, хто вижив, становить 28,6 року, а тих, хто загинув, — 30,4 року. Найвищий рівень виживання мали діти до 12 років (58 %), найнижчий — пасажири 19–30 років (33,1 %). Пік на гістограмі в інтервалі 25–30 років штучний: його утворили 177 пропусків, замінених середнім віком.

Кореляційний аналіз підтверджує ці висновки. Найсильніше з виживанням пов’язані стать (r = 0,54), клас (r = −0,34; що вищий номер класу, то нижче виживання) та тариф (r = 0,26). Зв’язок із віком майже відсутній (r = −0,07). Отже, шанси врятуватися визначали насамперед стать і соціальний статус пасажира.